# nb04 — 신구 교차검증, 일사 물리 검증, 운량 대안, 파일럿 수집

1. 같은 R030 모델을 구 grib API 와 신 std API 로 읽었을 때 값이 일치하는가
2. 신규 일사 3변수(SWDDIR2/SWDDIF2/SWDDNI2)의 물리적 의미와 ACSWDNB(누적) 규칙
3. 지역·국지에 없는 전운량의 대안
4. 파일럿: 신 API 로 운영 꼴의 wide 를 실제로 조립

In [1]:
import sys
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
import numpy as np
import pandas as pd
import probe_lib as pl
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

def std_val(grp, nwp, nm, tmfc, hf, x, y, data="U", level=None):
    """캐시된 신규 std 응답에서 첫 값 (캐시에 없으면 실호출 1회)."""
    b = pl.fetch_std(grp, nwp, nm, tmfc, hf, x, y, data=data, level=level)
    if not b or "ERROR" in b:
        return None
    v = [float(t) for l in b.splitlines()
         if l.strip() and not l.startswith("#") for t in l.split()]
    return v[0] if v else None

TMFC = "2026070312"   # 본 연구의 기준 발표 (2026-07-03 12z) -- 캐시 고정
print("호출 예산 상태:", pl.budget_status())

호출 예산 상태: {'real_calls': 1904, 'hard_cap': 10000, 'remaining': 8096}


In [2]:
# 1) 신구 R030 교차검증 (솔라팜 550,250 -- 신구 같은 격자점, hf 12~21)
cc = pd.read_csv("results/crosscheck_r030.csv")
agg = cc.groupby("var").agg(구API_평균=("value_old", "mean"), 신API_평균=("value_new", "mean"),
                            최대차이=("diff", lambda s: s.abs().max()))
agg.round(3)

,구API_평균,신API_평균,최대차이
var,,,
GUST,7.747,7.748,0.005
RH2,99.320,99.319,0.005
T2,296.005,296.007,0.005
TSKIN,295.564,295.563,0.005
U10,2.887,2.888,0.005
U80,4.149,4.149,0.005
V10,4.166,4.167,0.005
V80,4.920,4.919,0.005


**완전 일치** (최대 차이 0.005 = 2byte 패킹 양자화 한계). 신 std API 의 지역·국지 값은 그대로
신뢰할 수 있고, 바람 성분도 회전 없이 일치한다. — 전구의 화면고도 버그(nb02)와 대조적.

In [3]:
# 2) 일사 삼각검증 (서산 L010, 2026-07-04 하루)
sq = pd.read_csv("results/solar_seq.csv")
day = sq[sq.cosZ > 0.05].copy()
day["SWDDIR2/(DNIxcosZ)"] = day.SWDDIR2 / day.dni_cosz
print(day[["hf", "cosZ", "SWDDIR2", "SWDDIF2", "SWDDNI2", "dni_cosz", "ghi_sum",
           "SWDDIR2/(DNIxcosZ)"]].round(3).to_string(index=False))

 hf  cosZ  SWDDIR2  SWDDIF2  SWDDNI2  dni_cosz  ghi_sum  SWDDIR2/(DNIxcosZ)
  9 0.112    11.35    23.35   102.05    11.408    34.70               0.995
 10 0.304     5.85    95.55    19.30     5.870   101.40               0.997
 11 0.492    46.65   229.75    94.90    46.659   276.40               1.000
 12 0.662   132.35   393.10   200.10   132.395   525.45               1.000
 13 0.802   765.40    56.20   954.00   765.544   821.60               1.000
 14 0.905   876.65    60.35   969.35   876.792   937.00               1.000
 15 0.961   938.30    62.60   976.70   938.474  1000.90               1.000
 16 0.968   946.05    62.90   977.95   946.314  1008.95               1.000
 17 0.924   898.70    61.15   972.50   898.995   959.85               1.000
 18 0.834   800.10    57.50   959.70   800.484   857.60               1.000
 19 0.703   657.80    51.90   936.60   658.285   709.70               0.999
 20 0.540   484.30    44.40   898.60   484.882   528.70               0.999
 21 0.355   

- `SWDDIR2 ≈ SWDDNI2 × cos(천정각)` (비율 1.00) → **SWDDIR2 = 수평면 직달 일사** 확정
- 따라서 **전천일사 GHI = SWDDIR2 + SWDDIF2** (직달수평 + 산란) — 현행 DB 규약(radiation, MJ/m²·h)으로는
  `GHI × 0.0036` 그대로 사용 (현행 dswrsfc 변환식과 동일)
- 덤: 법선면 직달(DNI)·산란 분리는 태양광 모델에 현행보다 **더 풍부한** 입력이다.

In [4]:
# ACSWDNB(누적 일사, MJ/m^2) 규칙: 발표 기준 누적(리셋 없음), 시간 차분 = 시간당 에너지
print(sq[["hf", "ACSWDNB", "acswdnb_diff_MJ", "ghi_sum_MJ"]].round(3).to_string(index=False))
print()
print("hf=0 에서 0, 단조 증가, 야간 증가 0 -> 강수 누적(rainc_acc) diff 패턴 재사용 가능")

 hf  ACSWDNB  acswdnb_diff_MJ  ghi_sum_MJ
  0     0.00              NaN       0.000
  1     0.00             0.00       0.000
  2     0.00             0.00       0.000
  3     0.00             0.00       0.000
  4     0.00             0.00       0.000
  5     0.00             0.00       0.000
  6     0.00             0.00       0.000
  7     0.00             0.00       0.000
  8     0.00             0.00       0.000
  9     0.04             0.04       0.125
 10     0.30             0.26       0.365
 11     1.02             0.72       0.995
 12     2.63             1.61       1.892
 13     4.93             2.30       2.958
 14     8.02             3.09       3.373
 15    11.45             3.43       3.603
 16    14.74             3.29       3.632
 17    18.22             3.48       3.455
 18    21.43             3.21       3.087
 19    24.18             2.75       2.555
 20    26.34             2.16       1.903
 21    27.82             1.48       1.181

hf=0 에서 0, 단조 증가, 야간 증가 0 -> 강수 누

In [5]:
# 3) 운량 대안: R030 등압면 CLDFRA(레벨별) 결합 vs 전구 tcld
ca = pd.read_csv("results/cloud_alternative.csv")
ca

,point,hf,cldfra_total_randov,cldfra_total_maxov,cldfra_low_max,cldfra_mid_max,cldfra_high_max,ne57_tcld,ne57_lcld,ne57_mcld,ne57_hcld
0,solar_farm(비),12,1.000,1.00,0.56,0.84,1.00,1.00,0.33,0.99,0.62
1,solar_farm(비),15,0.978,0.71,0.70,0.71,0.00,0.40,0.02,0.24,0.18
2,solar_farm(비),18,1.000,1.00,0.00,1.00,0.02,1.00,0.48,1.00,0.99
3,Seosan(맑음),12,0.000,0.00,0.00,0.00,0.00,1.00,1.00,0.08,0.26
4,Seosan(맑음),15,0.000,0.00,0.00,0.00,0.00,0.85,0.23,0.01,0.81
5,Seosan(맑음),18,0.000,0.00,0.00,0.00,0.00,0.97,0.53,0.00,0.97


**운량 판정**
- 지역·국지 신모델 산출물에는 운량 단일면 변수가 **없다** (문서의 LCDC/MCDC/HCDC 는 실제 파일에 미탑재,
  구 grib 의 TCOG/TCOH 도 0 고정 = 사실상 사망).
- 등압면 CLDFRA 레벨 결합(랜덤 오버랩)은 물리적으로 정합(비 오는 제주 ≈ 1.0, 맑은 서산 = 0.0 —
  같은 모델의 일사 예측과도 일치)하지만 시각당 18콜이라 운영 비용이 크다.
- **현행 구조가 이미 정답**: 운량(tcld/mcld/lcld)은 지금도 전구(KIMG)에서 수집한다. 지역·국지를 도입해도
  운량은 전구 병합을 유지하면 되고, 필요 시 CLDFRA 는 EDA·특수 분석용으로만 쓴다.

In [6]:
# 4) 파일럿: 신 std 로 수집한 wide 샘플 (솔라팜 D+1 하루, 240건 240 성공, 3.3콜/s)
w = pd.read_parquet("pilot_wide_sample.parquet")
w[["temp_C", "RH2", "U10", "V10", "GUST", "ghi", "radiation_MJ", "ACSWDNB"]].round(2)

,temp_C,RH2,U10,V10,GUST,ghi,radiation_MJ,ACSWDNB
timestamp,,,,,,,,
2026-07-04 00:00:00,21.96,90.80,-1.00,0.09,1.77,0.00,0.00,0.00
2026-07-04 01:00:00,21.51,95.56,-0.92,-0.80,2.62,0.00,0.00,0.00
2026-07-04 02:00:00,21.41,99.55,-1.17,0.33,4.68,0.00,0.00,0.00
2026-07-04 03:00:00,21.70,100.00,-1.04,1.11,3.70,0.00,0.00,0.00
2026-07-04 04:00:00,21.80,100.00,-1.30,1.18,2.78,0.00,0.00,0.00
2026-07-04 05:00:00,22.04,100.00,-0.87,1.59,3.15,0.00,0.00,0.00
2026-07-04 06:00:00,22.33,100.00,0.13,1.74,2.18,25.15,0.09,0.01
2026-07-04 07:00:00,23.04,98.44,0.63,0.31,1.15,147.50,0.53,0.22
2026-07-04 08:00:00,23.87,96.98,-0.01,0.69,0.80,289.75,1.04,0.94


## ★발견 2 — 구 pt 엔드포인트가 세 모델을 전부 지원한다

`nph-kim_nc_pt_txt2`(현행 전구 수집기가 쓰는 그 엔드포인트)에 `group=KIMR&nwp=R030` 또는
`group=KIML&nwp=L010` 을 주면 **지역·국지도 그대로 응답**한다. 게다가:
- 위경도 직접 입력 (격자 변환 불필요 — X/Y 를 응답 헤더로 돌려주는데 우리 변환표와 일치)
- **콤마 멀티변수 = 1콜** (신 std 는 변수당 1콜 — 같은 일을 하려면 콜 수 10배 이상)
- 신규 변수(U140/U220, SWDDIR2/SWDDIF2/SWDDNI2, ACSWDNB, TSKIN, PBLH, VIS, RAIN)도 전부 나옴

즉 **기존 수집 엔진(_common.py 의 fetch_one_hf 패턴)에서 group/nwp/name 만 바꾸면 지역·국지 확장이
끝난다.** 신 std API 는 격자 박스 추출(map=S+sub) 같은 특수 용도 외에는 쓸 이유가 없다.

In [7]:
# 증거: 구 pt 로 L010 확장 변수 멀티 호출 (1콜)
b = pl.fetch(pl.URL_OLD_PT, {"group": "KIML", "nwp": "L010", "data": "U",
    "name": "U140,V140,U220,V220,TSKIN,PBLH,VIS,SWDDNI2,RAIN",
    "tmfc": TMFC, "hf": "18", "lat": "33.3868", "lon": "126.8802", "disp": "A", "help": "0"})
for ln in b.splitlines():
    if ln.strip() and not ln.startswith("#"):
        print(ln)

2026070312 2026070406       33        0  5.11000e+00  U140(m s-1)
2026070312 2026070406       36        0  3.22000e+00  V140(m s-1)
2026070312 2026070406       34        0  7.72000e+00  U220(m s-1)
2026070312 2026070406       37        0  5.21000e+00  V220(m s-1)
2026070312 2026070406       12        0  3.01290e+02  TSKIN(K)
2026070312 2026070406       16        0  1.83500e+02  PBLH(m)
2026070312 2026070406       25        0  1.00001e+05  VIS(m)
2026070312 2026070406       40        0  1.77500e+01  SWDDNI2(W m-2)
2026070312 2026070406       17        0  6.63000e+00  RAIN(mm)


In [8]:
# 마무리: 이번 연구의 콜 결산
log = pd.read_csv("probe_cache/calls_log.csv")
real = log[log.cached == 0]
print(f"실호출 {len(real)}건 / 캐시히트 {len(log) - len(real)}건 (하드캡 10,000)")
print(real.groupby("endpoint").size().to_string())

실호출 1904건 / 캐시히트 178건 (하드캡 10,000)
endpoint
kim_grib_pt_tmfc.php         4
nph-kim_nc_pt_txt2          21
nph-kim_nc_xy_txt2_std    1875
nph-nwp_latlon_api           4
